### np.transpose

In [7]:
def get_shape(a):
    if not isinstance(a , list): 
        return ()
    if len(a) == 0: 
        return (0 , )
    inner_shape = get_shape(a[0])

    for i in range(1 , len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected : the shape of first row is {inner_shape}" , 
                f"but row {i} has shape {get_shape(a[i])}"
            )

    return (len(a) , ) + inner_shape

def get_element(a , indices): 
    depth = 0

    for idx in indices:
        if not isinstance(a , list): 
            raise TypeError(f"expected a list at depth {depth} , but git {type(a).__name__}")
        if idx >= len(a) or idx < -len(a):
            raise IndexError(f"at depth {depth} index {idx} is out of range for {len(a)}")

        a = a[idx]
        depth += 1

    return a 

def set_element(a, indices, value):
    depth = 0
    for idx in indices[:-1]:
        if not isinstance(a, list):
            raise TypeError(f"at depth {depth}: expected list, got {type(a).__name__}")

        if idx >= len(a) or idx < -len(a):
            raise IndexError(f"at depth {depth}: index {idx} out of range")

        a = a[idx]
        depth += 1

    last_idx = indices[-1]

    if not isinstance(a, list):
        raise TypeError("too many indices")

    if last_idx >= len(a) or last_idx < -len(a):
        raise IndexError(f"final index {last_idx} out of range")

    a[last_idx] = value

def deep_copy(a):
    if isinstance(a , list):
        return [deep_copy(item) for item in a]
    return a  

def np_zeroes(a , dtype = float):
    if not isinstance(a,tuple):
        raise TypeError("input must be of type tuple")
    for dim in a:
        if not isinstance(dim , int):
            raise TypeError(f"{dim} is not an integer")
        if dim < 0: 
            raise ValueError(f"{dim} is negative")
    if not callable(dtype):
        raise TypeError(f"dtype must be callable but got {type(dtype).__name__}")
    zero = dtype(0)
    if len(a) == 0: 
        return zero
    if len(a) == 1: 
        return [zero] * a[0]
    else:
        result = []
        for i in range(a[0]):
            result.append(np_zeroes(a[1:] , dtype))
        return result



In [8]:
import itertools

def np_transpose(a, axes=None):
   
    if not isinstance(a, list):
        raise TypeError(f"expected a list, got {type(a).__name__}")

    shape = get_shape(a)
    ndim = len(shape)

    if ndim <= 1:
        return deep_copy(a)

    if axes is None:
        axes = tuple(range(ndim - 1, -1, -1))
    else:
        if isinstance(axes, list):
            axes = tuple(axes)
        if not isinstance(axes, tuple):
            raise TypeError(f"axes must be a tuple or list, got {type(axes).__name__}")
        # normalize negative axes
        axes = tuple(ax if ax >= 0 else ndim + ax for ax in axes)
        if sorted(axes) != list(range(ndim)):
            raise ValueError(f"axes {axes} is not a valid permutation of {tuple(range(ndim))}")

    new_shape = tuple(shape[ax] for ax in axes)

    result = np_zeroes(new_shape, dtype=int)   

    for combo in itertools.product(*[range(s) for s in new_shape]):
     
        old_idx = [None] * ndim
        for new_ax, old_ax in enumerate(axes):
            old_idx[old_ax] = combo[new_ax]

        val = get_element(a, old_idx)
        set_element(result, list(combo), val)

    return result

### testcase

In [9]:
print(np_transpose([[1,2,3],[4,5,6]]))              
print(np_transpose([[1,2,3],[4,5,6]], (1,0)))       

a = [[[1,2],[3,4]],[[5,6],[7,8]]]
print(np_transpose(a))                           
print(np_transpose(a, (1,0,2)))                    
print(np_transpose([1,2,3]))     

[[1, 4], [2, 5], [3, 6]]
[[1, 4], [2, 5], [3, 6]]
[[[1, 5], [3, 7]], [[2, 6], [4, 8]]]
[[[1, 2], [5, 6]], [[3, 4], [7, 8]]]
[1, 2, 3]


In [10]:
np_transpose([[1,2],[3,4]], (0,0))

ValueError: axes (0, 0) is not a valid permutation of (0, 1)

In [11]:
np_transpose([[1,2],[3,4]], (0,))

ValueError: axes (0,) is not a valid permutation of (0, 1)

In [12]:
np_transpose(5)

TypeError: expected a list, got int